# EXP-006 — M5 Hybrid GraphCodeBERT + CPG-GNN (GPU)

Notebook train mô hình hybrid: **text encoder = GraphCodeBERT (M2) + graph encoder = CPG-GNN (M4)** theo DEC-029. Dùng lại **exact CPG export/cache của EXP-005** (không parse Joern lại), warm start encoder từ checkpoint M2 (EXP-003) và M4 (EXP-005), rồi joint fine-tune LR 2e-5.

**Warm-start checkpoints được lấy từ Hugging Face Hub** — không thể nằm trong repo vì `.gitignore` loại mọi `*.pt` (M2 nặng 1.41 GB). Hai file `m2_text_encoder.pt` (~0.5 GB) + `m4_graph_encoder.pt` (vài MB) được sinh bằng `scripts/export_hybrid_warm_start.py` ở máy cá nhân rồi push lên HF repo `kichirou244/sccd-hybrid-warm-start` (đổi ở cell 7). Nếu repo **private** thì cần **Internet = ON** + `HF_TOKEN` (cell 1) là pull được. Kèm fallback: nếu trước đó có Add input output EXP-003/EXP-005 thì cell 7 cũng quét `/kaggle/input`. **Thiếu checkpoint ⇒ cell 7 assert chặn train** — tránh báo cáo số không chính thức.

**Chạy bằng GPU** (T4/P100). Job dài nên: chạy tay cell 2→9 trước (toàn bước nhanh, ~20 phút), rồi **Save & Run All** cho cell 10 (train full) — đừng chạy tay cell 10 rồi giữ tab.

**Trước khi chạy:** Add input dataset shards M4 (tên ở cell 4). Warm start chạy qua HF nên **không cần** thêm input output EXP-003/EXP-005.

### Thứ tự chạy
1. Cell 2 (**clone hoặc pull** code mới) → cell 3 (pip torch_geometric)
2. Cell 4 (nạp shards CPG) → cell 5 (dựng pairs)
3. Cell 6 PREFLIGHT → cell 7 WARM-START CHECKPOINTS (HF pull) → cell 8 SMOKE → cell 9 DIAGNOSTIC (gate `logit std > 0` + xác nhận warm start loaded)
4. Cell 10 FULL TRAIN (5 epoch trần, early stopping patience 2 — tự evaluate khi xong)
5. Cell 11 EVALUATE lại + kiểm `EVAL_COMPLETE.json`
6. **Save Version** → gửi `metrics.json` về

In [ ]:
print(">>> CELL 1: secrets + helpers", flush=True)
import json
import os
import subprocess
import sys
from pathlib import Path

# Kaggle Add-ons -> Secrets -> GITHUB_TOKEN + HF_TOKEN.
# HF_TOKEN chi can cho dataset POJ-104 tu HF khi chua co o data/ (cell 5 build lai).
GITHUB_TOKEN = ""
HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret("GITHUB_TOKEN")
    HF_TOKEN = HF_TOKEN or _s.get_secret("HF_TOKEN")
except Exception as _e:
    print("[secrets] khong doc duoc secrets:", _e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
if GITHUB_TOKEN:
    os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")
print("[secrets] github =", bool(GITHUB_TOKEN), "| hf =", bool(HF_TOKEN))

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f"$ {cmd}\n{r.stdout}{r.stderr}")
    if r.returncode != 0:
        raise RuntimeError(f"lenh that bai (rc={r.returncode}): {cmd}")
    return r.stdout

def sh_live(cmd, **kw):
    """Nhu sh() nhung STREAM stdout theo dong, thay vi doi process xong.
    Dung cho cell chay lau (train/evaluate): khong co la 2-3h im lang."""
    import subprocess as _sp
    print(f"$ {cmd}", flush=True)
    p = _sp.Popen(cmd, shell=True, stdout=_sp.PIPE, stderr=_sp.STDOUT,
                  text=True, bufsize=1, **kw)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"lenh that bai (rc={rc}): {cmd}")
    return ""


In [ ]:
print(">>> CELL 2: clone/pull repo", flush=True)
REPO = "kichirou244/SemanticCodeCloneDetection"
assert REPO != "YOUR_USER/YOUR_REPO", "Sua REPO truoc khi chay cell tiep theo"

if Path(".git").exists():
    # Da clone o session truoc => **phai pull** — khong pull la chay code cu.
    sh('git pull --ff-only || (git fetch --all --depth 1 && '
       'git reset --hard origin/HEAD)')
else:
    url = f'https://github.com/{REPO}.git'
    if GITHUB_TOKEN:
        url = url.replace('https://', f'https://x-access-token:{GITHUB_TOKEN}@')
    sh(f'git clone --depth 1 {url} repo')
    sh('mv repo/* repo/.[!.]* . 2>/dev/null; rmdir repo')
sh('git log --oneline -1 && python -V && nproc')


In [ ]:
print(">>> CELL 3: pip torch_geometric", flush=True)
# torch co san tren image GPU Kaggle. Chi cai them torch_geometric (GATConv)
# + transformers (GraphCodeBERT + scheduler — giong M1/M2/M3/M4).
import torch
print('torch', torch.__version__, '| cuda build:', torch.version.cuda,
      '| gpu:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
sh('pip install -q torch_geometric transformers')
print(sh('python -c "import torch_geometric as pyg; print(\'pyg\', pyg.__version__); '
         'from torch_geometric.nn import GATConv; print(\'GATConv OK\')"'))


In [ ]:
print(">>> CELL 4: nap dataset", flush=True)
# Nap dataset M4 — chay duoc voi CA dataset cu (folder bung san) LAN dataset
# shards moi. Truoc khi chay: Add input dataset (cu hoac shards deu duoc).
# Input la read-only nen COPY/BUNG ve ./artifacts (benchmark.json can ghi).
import glob
from pathlib import Path
sh('mkdir -p artifacts/cpg_cache')
_exp_path = Path('artifacts/cpg_export')
have = len(list(_exp_path.glob('*.json'))) if _exp_path.exists() else 0
shards = sorted(glob.glob('/kaggle/input/**/cpg_shard_*.jsonl', recursive=True))
print('shards:', len(shards), '| da co:', have)
if have >= 51000:
    print(f'da co ./artifacts ({have} JSON) — dung lai')
elif shards:
    from pathlib import PurePath as _PP
    sdirs = sorted({_PP(s).parent.as_posix() for s in shards})
    npzs = sorted(glob.glob('/kaggle/input/**/cpp_*_full.npz', recursive=True))
    print('bung tu', len(sdirs), 'thu muc shard + copy', len(npzs), 'npz ...')
    sh(f'python scripts/make_cpg_shards.py --from-shards {" ".join(sdirs)} '
       f'--out artifacts/cpg_export')
    for _nz in npzs:
        sh(f'cp {_nz} artifacts/cpg_cache/')
else:
    # Thu muc bung san: duong cua ban uu tien truoc, roi moi glob chung.
    _pref = Path('/kaggle/input/datasets/kichirou/'
                 'semanticcodeclonedetection-dataset-m4/artifacts')
    _exp = _pref if (_pref / 'cpg_export').exists() else None
    if _exp is None:
        _found = glob.glob('/kaggle/input/**/cpg_export', recursive=True)
        _exp = Path(_found[0]).parent if _found else None
    _zp = glob.glob('/kaggle/input/*/*.zip') + glob.glob('/kaggle/input/*.zip')
    sh('rm -rf artifacts/cpg_export artifacts/cpg_cache && mkdir -p artifacts/cpg_cache')
    if _exp is not None:
        print('dung thu muc:', _exp)
        sh(f'cp -r {_exp}/cpg_export artifacts/ && cp -r {_exp}/cpg_cache artifacts/ '
           f'&& echo COPY_XONG')
    elif _zp:
        sh(f'unzip -q -o {_zp[0]} -d _m4in && echo UNZIP_XONG')
        cands = glob.glob('_m4in/**/cpg_export', recursive=True)
        assert cands, 'khong thay cpg_export trong zip'
        src = str(Path(cands[0]).parent)
        sh(f'cp -r {src}/cpg_export artifacts/ && cp -r {src}/cpg_cache artifacts/ '
           f'&& echo COPY_XONG')
    else:
        raise SystemExit('Chua add dataset input M4 — Add input truoc khi chay cell nay')
sh('ls artifacts/cpg_export | wc -l && ls artifacts/cpg_cache/ && du -sh artifacts/')


In [ ]:
print(">>> CELL 5: dung pairs", flush=True)
# Dung pairs + manifest + samples (can cho validate_dataset va train).
# Build lai bang dung script cua extraction — cung splits, cung leakage check.
sh('python scripts/build_dataset.py --config configs/pairs/poj104.yaml')
print(sh('python -c "import json; m=json.load(open(\'data/manifests/poj104.json\')); '
         'print(\'leakage_passed =\', m[\'leakage_passed\'])"'))
sh('wc -l data/processed/poj104/samples.jsonl data/pairs/poj104/*.jsonl')


In [ ]:
print(">>> CELL 6: PREFLIGHT", flush=True)
# PREFLIGHT — gate truoc khi ton gio GPU: schema 6 loai + du export/cache/pairs + VRAM.
sh('python -m src.experiments.exp006_hybrid --config configs/exp006_hybrid.yaml '
   '--mode preflight --device cuda')


In [ ]:
print(">>> CELL 7: WARM-START CHECKPOINTS (M2 + M4) - Hugging Face pull", flush=True)
# M5 khoi tao encoder tu M2 (GraphCodeBERT, EXP-003) va M4 (CPG-GNN, EXP-005).
# Checkpoint KHONG o trong repo (gitignore *.pt). Nguon chuan: Hugging Face Hub.
# Sinh 2 file nay dung:
#   python scripts/export_hybrid_warm_start.py \
#       --m2 results/exp003_graphcodebert_poj104/best_checkpoint.pt \
#       --m4 results/exp005_cpg_gnn_poj104/best_checkpoint.pt \
#       --out warm_start \
#       --hf-repo kichirou244/sccd-hybrid-warm-start
HF_WARM_REPO = "kichirou244/sccd-hybrid-warm-start"  # khac thi doi cho nay

# Neu repo private -> nho HF_TOKEN (cell 1). Public -> khong can token.
_WARM_FILES = {
    "m2_text_encoder.pt": "results/exp003_graphcodebert_poj104/best_checkpoint.pt",
    "m4_graph_encoder.pt": "results/exp005_cpg_gnn_poj104/best_checkpoint.pt",
}
_LEGACY_TAGS = {  # fallback: quet output EXP-003/EXP-005 cu trong /kaggle/input
    "m2_text_encoder.pt": "exp003_graphcodebert_poj104",
    "m4_graph_encoder.pt": "exp005_cpg_gnn_poj104",
}

import glob as _glob
import os as _os
import shutil as _shutil
from pathlib import Path as _P


def _pull_hf(fname: str, dst: _P) -> bool:
    from huggingface_hub import hf_hub_download
    tok = _os.environ.get("HF_TOKEN") or None
    p = hf_hub_download(repo_id=HF_WARM_REPO, filename=fname, token=tok)
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"[warm] HF pull {HF_WARM_REPO}/{fname} ({_P(p).stat().st_size/2**20:.0f} MB) -> {dst}")
    _shutil.copy2(p, dst)
    return True


def _pull_legacy(fname: str, dst: _P) -> bool:
    tag = _LEGACY_TAGS[fname]
    cands = sorted(
        _glob.glob(f"/kaggle/input/**/{tag}/best_weights_only.pt", recursive=True)
        + _glob.glob(f"/kaggle/input/**/{tag}/best_checkpoint.pt", recursive=True))
    if not cands:
        return False
    src = _P(cands[0])
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"[warm] fallback copy {src} -> {dst} ({src.stat().st_size/2**30:.2f} GB)")
    _shutil.copy2(src, dst)
    return True


for _fname, _rel in _WARM_FILES.items():
    _dst = _P(_rel)
    if _dst.exists() and _dst.stat().st_size > 0:
        print(f"[warm] da co {_dst} - giu nguyen")
        continue
    try:
        _pull_hf(_fname, _dst)
    except Exception as _e:
        print(f"[warm] HF loi ({_e}) - thu fallback /kaggle/input")
        if not _pull_legacy(_fname, _dst):
            print(f"[warm] fallback khong co {_fname}")

_missing = [f for f, r in _WARM_FILES.items()
            if not (_P(r).exists() and _P(r).stat().st_size > 0)]
print("WARM_START:", "READY" if not _missing else f"MISSING {_missing}")
assert not _missing, (
    "Thieu checkpoint warm start. Kiem lai HF_WARM_REPO va HF_TOKEN (neu repo "
    "private), hoac Add input output EXP-003/EXP-005, roi chay lai cell 7.")


In [ ]:
print(">>> CELL 8: SMOKE", flush=True)
# SMOKE — 64 cap, 1 epoch, vai phut. Kiem VRAM voi dung batch_size train
# (batch 12 cap x ~350 node + text 256 token). Tren T4 qua VRAM full thi giam
# 'batch_size' trong config. Do xong la dung lai.
sh('python -m src.experiments.exp006_hybrid --config configs/exp006_hybrid.yaml '
   '--mode smoke --device cuda')


In [ ]:
print(">>> CELL 9: DIAGNOSTIC", flush=True)
# DIAGNOSTIC — grad/logit-std/throughput that + CHAY LA warm start (in
#   {"warm-text": {... "loaded", "missing"}...}) — missing > 0 la that bai.
# Phai in `logit std > 0`; scores hoi tu 0 => dung lai.
# `pairs_per_second_diag` de uoc gio train: 353.000 cap / toc do / 3600 = gio/epoch.
sh('python -m src.experiments.exp006_hybrid --config configs/exp006_hybrid.yaml '
   '--mode diagnostic --device cuda')


In [ ]:
print(">>> CELL 10: FULL TRAIN", flush=True)
# FULL TRAIN — 5 epoch tran, early stopping (patience 2) tu dung; xong tuong
# goi evaluate. CHAY BANG Save & Run All, dung chay tay giu tab.
# Melu y: nhin dong DAU cua log — phai co
#   {"warm-text": {... "loaded": true ...}} va {"warm-graph": {... "loaded": true ...}}
# Neu bat dau bang "[warm-*] bo qua" => thieu checkpoint (quay lai cell 7).
TIME_BUDGET_S = 11 * 3600  # limit phien Kaggle 9h tru hao install/dataset
# PYTHONWARNINGS tat UserWarning cua torch_geometric (scatter fallback chay dung).
sh_live('PYTHONWARNINGS=ignore::UserWarning python -m src.experiments.exp006_hybrid '
        '--config configs/exp006_hybrid.yaml '
        f'--mode train --device cuda --time-budget-s {TIME_BUDGET_S}')


In [ ]:
print(">>> CELL 11: EVALUATE", flush=True)
# EVALUATE — chay lai tren Kaggle (nguon so chuan, giong cac EXP khac).
sh_live('python -m src.experiments.exp006_hybrid --config configs/exp006_hybrid.yaml '
        '--mode evaluate --device cuda')
OUT = 'results/exp006_hybrid_poj104'
sh(f'test -f {OUT}/EVAL_COMPLETE.json && echo EVAL_COMPLETE || echo EVAL_CHUA_XONG')
print(sh(f'cat {OUT}/metrics.json'))
print(sh(f'cat {OUT}/threshold.json'))
print(sh('tail -n 8 {OUT}/training_history.csv'))


### Save Version — nhưng **chỉ** khi cell 11 in `EVAL_COMPLETE`

Version đã lưu **là kết quả M5**: `results/exp006_hybrid_poj104` (metrics, threshold, predictions, checkpoints). Model nhỏ (encoder 2 bản + head 3,1M) nên giữ checkpoint trong version.

**Trước khi coi số là chính thức, bắt buộc xác nhận** log train cell 10 có `{"warm-text": {...}}` và `{"warm-graph": {...}}` với `"missing": 0` — nếu không thì đây là pretrained/scratch, phải bỏ.

**Gửi về:** metrics.json, threshold.json, đuôi training_history.csv. Khi có số:
1. So sánh với **M2 0,9069** (test F1); kỳ vọng M5 > 0,9069 nhưng **đối chiếu cả ROC-AUC** (audit logistic baseline: F1 0,9217 nhưng AUC 0,9683 < M2 0,9713 — thắng chỉ ở vùng ngưỡng, không được kết luận chung chung, xem DEC-029).
2. Cập nhật registry EXP-006 (COMPLETED) + decision log + bảng kết quả báo cáo.

In [ ]:
print(">>> CELL 12: NEN KET QUA & DON DEP", flush=True)
import os
OUT = "results/exp006_hybrid_poj104"
# Xoa checkpoint warm start (chung chi la seed, da nap vao model roi) de zip nhe,
# chi giu ket qua M5.
for _keep in ("exp003_graphcodebert_poj104", "exp005_cpg_gnn_poj104"):
    print(f"xoa {_keep}...")
    sh(f"rm -rf results/{_keep}")
if os.path.exists(OUT):
    sh(f"zip -q -r {OUT}.zip {OUT}")
    print(f"\nDa nen xong: {OUT}.zip.")
else:
    print(f"\nChua co thu muc ket qua ({OUT}).")
print("\nDang xoa artifacts...")
sh("rm -rf artifacts")
print("Da don dep xong!")
